In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

In [ ]:
# sfs data
data_name = "NCI"

sfs_path = f"./data/processed/{data_name}/{data_name}_SFS_selected.csv"
df_sfs = pd.read_csv(sfs_path)

X_sfs = df_sfs.iloc[:,1:]
y_sfs = df_sfs.iloc[:,0]

# fs data
sksfs_path = f"./data/processed/{data_name}/{data_name}_SKSFS_5features.csv"
df_sksfs = pd.read_csv(sksfs_path)

X_fs = df_sksfs.iloc[:,1:]
y_fs = df_sksfs.iloc[:,0]

In [ ]:
dt = DecisionTreeClassifier(max_depth=5, random_state=42)
logi = LogisticRegression(max_iter=3000, random_state=42)

In [ ]:
method = ['sfs', 'fs']
results = {}

scaler = StandardScaler()

for m in method:
    # Chọn bộ data tương ứng
    if m == 'sfs':
        X_cur, y_cur = X_sfs, y_sfs
    else:
        X_cur, y_cur = X_fs, y_fs

    X_train, X_test, y_train, y_test = train_test_split(X_cur, y_cur,test_size=0.2, random_state=42, stratify=y_cur)

    # scale X
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)


    # Train model
    dt.fit(X_train, y_train)
    logi.fit(X_train_scaled, y_train)
    
    # Predict & Tính Accuracy
    y_dt_pred = dt.predict(X_test)
    acc_dt = accuracy_score(y_test, y_dt_pred)
    
    y_log_pred = logi.predict(X_test_scaled)
    acc_log = accuracy_score(y_test, y_log_pred)

    log_pipeline = make_pipeline(StandardScaler(), logi)

    # cv scores
    cv_dt = cross_val_score(dt, X_cur, y_cur, cv=4, scoring='accuracy')
    cv_log = cross_val_score(log_pipeline, X_cur, y_cur, cv=4, scoring='accuracy')
    
    dt_cv = round(cv_dt.mean(), 4)
    log_cv = round(cv_log.mean(), 4)
    results[m] = {
        'DecisionTreeClassifier': acc_dt,
        'LogisticRegression': acc_log,
        'DecisionTreeClassifier_cv': dt_cv,
        'LogisticRegression_cv': log_cv,
    }

    print(f" Method {m.upper()}: \n")
    print(f"  LogisticRegression  : hold-out={acc_log:.4f}  |  CV={log_cv}")
    print(f"  DecisionTree        : hold-out={acc_dt:.4f}  |  CV={dt_cv}\n")

## Chart + Report

### Chart

In [ ]:
labels = ['DT (Hold-out)', 'DT (CV)', 'Logi (Hold-out)', 'Logi (CV)']

# Lấy điểm của bộ SFS (All best features) và FS (5 features)
sfs_scores = [
    results['sfs']['DecisionTreeClassifier'], 
    results['sfs']['DecisionTreeClassifier_cv'],
    results['sfs']['LogisticRegression'], 
    results['sfs']['LogisticRegression_cv']
]

fs_scores = [
    results['fs']['DecisionTreeClassifier'], 
    results['fs']['DecisionTreeClassifier_cv'],
    results['fs']['LogisticRegression'], 
    results['fs']['LogisticRegression_cv']
]

In [ ]:
x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10,6))

# Vẽ 2 cột đứng cạnh nhau. Màu Nord "hơi bị chill" luôn nha!
rects1 = ax.bar(x - width/2, sfs_scores, width, label='SFS (Best Features(5))', color='#5E81AC')
rects2 = ax.bar(x + width/2, fs_scores, width, label='SKSFS (5 Features)', color='#A3BE8C')

# Trang trí cho nó "keo lỳ"
ax.set_ylabel('Accuracy Score', fontsize=12)
ax.set_title(f'So sánh Model & Feature Selection ({data_name} Dataset)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim([0, 1.15]) #type: ignore
ax.legend(loc='upper right')

# Hiện số % chính xác trên đầu mỗi cột
ax.bar_label(rects1, padding=3, fmt='%.4f')
ax.bar_label(rects2, padding=3, fmt='%.4f')

fig.tight_layout()

# Lưu cái hình lại để mốt quăng vô slide
img_path = f"./result/{data_name}/plot/sfs_comparison_chart.png"
plt.savefig(img_path, dpi=300) # dpi=300 cho nét căng
plt.show()

In [ ]:
# Report
report_path = f"result/{data_name}/report/sfs_comparing.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"󰈙 SFS vs FS (sklearn) comparing report.\n")
    f.write("-"*60+"\n")
    f.write(f"  [Decision Tree]\n")
    for m in method:
        f.write(f"    - Hold-out Accuracy : {results[m]['DecisionTreeClassifier']:.4f}\n")
        f.write(f"    - Cross-Val Accuracy: {results[m]['DecisionTreeClassifier_cv']:.4f}\n\n")
        f.write(f"  [Logistic Regression]\n")
        f.write(f"    - Hold-out Accuracy : {results[m]['LogisticRegression']:.4f}\n")
        f.write(f"    - Cross-Val Accuracy: {results[m]['LogisticRegression_cv']:.4f}\n")
        f.write("-" * 35 + "\n\n")

    f.write(" Note: Hold-out là test trên 20% data chia ra. CV là test chéo 4 Folds.\n")